In [27]:
import xgboost as xgb
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

In [28]:
df = pd.read_csv("../../data/EcoCrop_DB_cleaned.csv")

In [29]:
numeric_cols = [
    "GMIN",
    "GMAX",
    "TOPMN",
    "TOPMX",
    "TMIN",
    "TMAX",
    "ROPMN",
    "ROPMX",
    "RMIN",
    "RMAX",
    "PHOPMN",
    "PHOPMX",
    "PHMIN",
    "PHMAX",
]
categorical_cols = ["COMNAME"]

In [30]:
x = df[categorical_cols + numeric_cols]
y = df["Soil_Impact"]

In [31]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

In [33]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [35]:
model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            xgb.XGBClassifier(
                n_estimators=200,
                learning_rate=0.5,
                max_depth=5,
                random_state=42,
                eval_metric="mlogloss",
            ),
        ),
    ]
)

In [36]:
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['COMNAME','GMIN','GMAX',...,'PHOPMX','PHMIN','PHMAX']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining 

In [37]:
y_pred = model_pipeline.predict(X_test)

In [38]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.53

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.36      0.47        11
           1       0.44      0.70      0.54        10
           2       0.62      0.56      0.59         9

    accuracy                           0.53        30
   macro avg       0.58      0.54      0.53        30
weighted avg       0.58      0.53      0.53        30



In [39]:
param_grid = {
    "classifier__learning_rate": [0.01, 0.4, 0.5, 0.1, 0.2],
    "classifier__max_depth": [3, 5, 7],
    "classifier__n_estimators": [50, 100, 200],
}

In [40]:
grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,  # if 0 will not print the progress of learning
)

In [41]:
print("searching for best params")

grid_search.fit(X_train, y_train)

searching for best params
Fitting 5 folds for each of 45 candidates, totalling 225 fits


KeyboardInterrupt: 

In [ ]:
print("the best params")
print(grid_search.best_params_)

the best params
{'classifier__learning_rate': 0.5, 'classifier__max_depth': 5, 'classifier__n_estimators': 200}


In [ ]:
print(
    f"best accuracy: {grid_search.best_score_:.2f}",
)

best accuracy: 0.52


In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
y_pred_best = best_model.predict(X_test)

In [ ]:
print(f"final accuracy: {accuracy_score(y_test, y_pred_best):.2f}")

final accuracy: 0.53


In [ ]:
print(classification_report(y_test, y_pred_best))

              precision    recall  f1-score   support

           0       0.67      0.36      0.47        11
           1       0.44      0.70      0.54        10
           2       0.62      0.56      0.59         9

    accuracy                           0.53        30
   macro avg       0.58      0.54      0.53        30
weighted avg       0.58      0.53      0.53        30



In [ ]:
import os
import time
from pathlib import Path

import ee
import pandas as pd

START = "2016-01-01"
END = "2025-12-31"
EGYPT_LOCATIONS = {
    "beheira_damanhur": [30.47, 31.04],
    "nubaria_reclamation": [
        30.07,
        30.67,
    ],
    "kafr_el_sheikh": [30.94, 31.11],  # Core rice, sugar beet & cotton belt
    "dakahlia_mansoura": [31.38, 31.04],  # Central-east Delta intensive agriculture
    "gharbia_tanta": [31.00, 30.79],  # Central Delta (potatoes, flax, citrus, wheat)
    "menoufia": [31.00, 30.50],  # Southern Delta intensive horticulture
    "sharqia": [31.50, 30.60],  # Eastern Delta & Salhia desert reclamation
    "qalyubia_banha": [31.18, 30.46],  # Southern Delta orchards & vegetables
    "damietta": [31.81, 31.42],  # Northern coastal Delta (rice, guava, date palm)
    "ismailia": [32.27, 30.60],  # Suez Canal zone (mangoes, citrus, peanuts)
    # --- North Coast & Sinai ---
    "alexandria_burg_el_arab": [
        29.55,
        30.88,
    ],  # Rainfed/supplemental figs, olives & barley
    "matrouh_siwa": [25.52, 29.20],  # Oasis agriculture (dates, olives)
    "north_sinai_arish": [33.80, 31.13],  # Coastal olives, peaches & almonds
    # --- Middle Egypt ---
    "giza_badrashin": [31.27, 29.85],  # Peri-urban vegetables & date palms
    "fayoum": [30.84, 29.31],  # Oasis depression (medicinal plants, olives, grains)
    "beni_suef": [31.10, 29.07],  # Aromatic/medicinal herbs, maize & wheat
    "minya": [30.75, 28.10],  # Sugarcane transition, grapes, garlic & wheat
    "asyut": [31.18, 27.18],  # Pomegranates, cotton, sorghum & wheat
    # --- Upper Egypt ---
    "sohag": [31.70, 26.56],  # Onions, sorghum, wheat & sugarcane
    "qena": [32.72, 26.16],  # Major sugarcane & banana belt
    "luxor": [32.64, 25.69],  # Sugarcane, tomatoes & winter vegetables
    "aswan_kom_ombo": [32.95, 24.47],  # High-heat sugarcane, dates & hibiscus
    # --- Western Desert & Mega-Reclamation Projects ---
    "new_valley_kharga": [30.55, 25.45],  # Groundwater-irrigated dates & alfalfa
    "new_valley_dakhla": [28.98, 25.50],  # Oasis wheat, alfalfa & orchards
    "new_valley_farafra": [27.97, 27.06],  # Desert reclamation grains & olives
    "toshka": [31.53, 22.60],  # Southern mega-reclamation (wheat, dates, alfalfa)
    "east_owainat": [28.72, 22.58],  # Center-pivot desert agriculture (wheat, potatoes)
}

LST = "MODIS/061/MOD11A2"
NDVI = "MODIS/061/MOD13Q1"
IMERG = "NASA/GPM_L3/IMERG_V07"
SMAP = "NASA/SMAP/SPL4SMGP/008"
ERA5 = "ECMWF/ERA5_LAND/DAILY_AGGR"
CACHE_DIR = Path("cache")
OUT_PATH = Path("../../data/egypt/egypt_climate_soil_data.csv")

FULL_DAYS = pd.date_range(START, END, freq="D")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def get_indicators(LOCATIONS, region, ERA5, cache_dir=CACHE_DIR, out_path=OUT_PATH):
    full_days = pd.DatetimeIndex(FULL_DAYS)
    start = full_days.min().normalize()
    end = full_days.max().normalize()
    end_excl = end + pd.Timedelta(days=1)

    cache_dir = Path(cache_dir)
    out_path = Path(out_path)
    cache_dir.mkdir(parents=True, exist_ok=True)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    def get_series(asset, bands, point, scale, months, retries=3):
        if isinstance(bands, str):
            bands = [bands]

        col_all = ee.ImageCollection(asset).select(bands)
        frames = []

        s = start
        while s < end_excl:
            e = min(s + pd.DateOffset(months=months), end_excl)
            data = None

            for attempt in range(retries):
                try:
                    data = (
                        col_all.filterDate(
                            s.strftime("%Y-%m-%d"), e.strftime("%Y-%m-%d")
                        )
                        .getRegion(point, scale)
                        .getInfo()
                    )
                    break
                except Exception as ex:
                    if attempt == retries - 1:
                        print(f"   ! skipped {s:%Y-%m} ({bands}): {ex}")
                    else:
                        time.sleep(3)

            s = e
            if not data or len(data) < 2:
                continue

            df = pd.DataFrame(data[1:], columns=data[0])
            frames.append(df[["time", *bands]])

        if not frames:
            return pd.DataFrame(columns=bands, index=pd.DatetimeIndex([]), dtype=float)

        df = pd.concat(frames)
        out = df[bands].apply(pd.to_numeric, errors="coerce")
        out.index = pd.to_datetime(df["time"], unit="ms")
        return out.sort_index()

    def daily(series, how="mean", interp_limit=None):
        if series.empty:
            return pd.Series(index=full_days, dtype=float)

        d = (
            series.resample("D").sum(min_count=1)
            if how == "sum"
            else series.resample("D").mean()
        )
        d = d.reindex(full_days)
        if interp_limit:
            d = d.interpolate(method="time", limit=interp_limit)
        return d

    all_frames = []

    for name, (lon, lat) in LOCATIONS.items():
        cache_file = cache_dir / f"{region}_{name}.csv"

        if cache_file.exists():
            print(f"\n=== {name}: loaded from cache ===")
            all_frames.append(
                pd.read_csv(cache_file, index_col="date", parse_dates=True)
            )
            continue

        print(f"\n=== {name} ({lon}, {lat}) ===")
        point = ee.Geometry.Point([lon, lat])

        print(" air temperature (ERA5-Land)...")
        era = (
            get_series(
                ERA5,
                ["temperature_2m", "temperature_2m_min", "temperature_2m_max"],
                point,
                11132,
                months=12,
            )
            - 273.15
        )

        print(" land surface temperature (MODIS LST)...")
        lst = get_series(LST, "LST_Day_1km", point, 1000, months=12) * 0.02 - 273.15

        print(" NDVI...")
        ndvi = get_series(NDVI, "NDVI", point, 250, months=12) * 0.0001

        print(" precipitation (half-hourly, slowest)...")
        precip = get_series(IMERG, "precipitation", point, 10000, months=1)
        precip = precip * 0.5
        precip = precip.where(precip >= 0)

        print(" soil moisture...")
        smap = get_series(SMAP, "sm_surface", point, 9000, months=1)

        df = pd.DataFrame(
            {
                "Temperature_C": daily(era["temperature_2m"], "mean"),
                "Temp_Min_C": daily(era["temperature_2m_min"], "mean"),
                "Temp_Max_C": daily(era["temperature_2m_max"], "mean"),
                "LST_Day_C": daily(lst["LST_Day_1km"], "mean", interp_limit=16),
                "NDVI": daily(ndvi["NDVI"], "mean", interp_limit=32),
                "Precipitation_mm": daily(precip["precipitation"], "sum"),
                "Soil_Moisture_m3m3": daily(smap["sm_surface"], "mean", interp_limit=3),
            }
        )
        df.index.name = "date"
        df["location"] = name
        df["lon"] = lon
        df["lat"] = lat
        df["month"] = df.index.month
        df["day_of_year"] = df.index.dayofyear

        sm_max = df["Soil_Moisture_m3m3"].max()
        if pd.notna(sm_max) and sm_max > 1:
            print(
                f"   ! WARNING: soil moisture max = {sm_max:.2f}, this is not m3/m3 "
                f"(probably mm). Check the SMAP asset/band in your config."
            )

        df.to_csv(cache_file)
        print(f" saved {cache_file} ({len(df)} rows)")
        all_frames.append(df)

    dataset = pd.concat(all_frames)
    dataset.to_csv(out_path)

    print("\nSample:")
    print(dataset.head())
    print("\nMissing values per column:")
    print(dataset.isna().sum())
    print(f"\nTotal rows: {len(dataset)}")


try:
    ee.Initialize(project="nasa-project-514716")
except Exception:
    print(
        "Earth Engine is not initialized. Run ee.Authenticate() once in a browser, "
        "then rerun this cell."
    )


get_indicators(EGYPT_LOCATIONS, "egypt", ERA5)


=== menoufia (31.0, 30.5) ===
 air temperature (ERA5-Land)...
 land surface temperature (MODIS LST)...
 NDVI...
 precipitation (half-hourly, slowest)...
 soil moisture...


C:\Users\AL aqsa\AppData\Local\Temp\ipykernel_11232\3316570064.py:158: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  df["day_of_year"] = df.index.dayofyear


 saved cache\egypt_menoufia.csv (3653 rows)

=== beheira_damanhur (30.47, 31.04) ===
 air temperature (ERA5-Land)...
 land surface temperature (MODIS LST)...
 NDVI...
 precipitation (half-hourly, slowest)...
 soil moisture...


C:\Users\AL aqsa\AppData\Local\Temp\ipykernel_11232\3316570064.py:158: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  df["day_of_year"] = df.index.dayofyear


 saved cache\egypt_beheira_damanhur.csv (3653 rows)

=== sharqia (31.5, 30.6) ===
 air temperature (ERA5-Land)...
 land surface temperature (MODIS LST)...
 NDVI...
 precipitation (half-hourly, slowest)...
 soil moisture...


C:\Users\AL aqsa\AppData\Local\Temp\ipykernel_11232\3316570064.py:158: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  df["day_of_year"] = df.index.dayofyear


 saved cache\egypt_sharqia.csv (3653 rows)

=== fayoum (30.84, 29.31) ===
 air temperature (ERA5-Land)...
 land surface temperature (MODIS LST)...
 NDVI...
 precipitation (half-hourly, slowest)...
 soil moisture...


C:\Users\AL aqsa\AppData\Local\Temp\ipykernel_11232\3316570064.py:158: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  df["day_of_year"] = df.index.dayofyear


 saved cache\egypt_fayoum.csv (3653 rows)

=== minya (30.75, 28.1) ===
 air temperature (ERA5-Land)...
 land surface temperature (MODIS LST)...
 NDVI...
 precipitation (half-hourly, slowest)...
 soil moisture...


C:\Users\AL aqsa\AppData\Local\Temp\ipykernel_11232\3316570064.py:158: Pandas4Warning: DatetimeIndex.dayofyear is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_year instead.
  df["day_of_year"] = df.index.dayofyear


 saved cache\egypt_minya.csv (3653 rows)

Sample:
            Temperature_C  Temp_Min_C  Temp_Max_C  LST_Day_C      NDVI  \
date                                                                     
2016-01-01      11.500101    8.754205   14.655008    16.9700  0.679600   
2016-01-02      11.594276    8.485422   15.548227    17.1975  0.674325   
2016-01-03      13.144283    9.517343   17.324426    17.4250  0.669050   
2016-01-04      14.668816    9.970895   20.120065    17.6525  0.663775   
2016-01-05      15.356809    9.839197   21.736063    17.8800  0.658500   

            Precipitation_mm  Soil_Moisture_m3m3  location   lon   lat  month  \
date                                                                            
2016-01-01               0.0            0.152622  menoufia  31.0  30.5      1   
2016-01-02               0.0            0.140373  menoufia  31.0  30.5      1   
2016-01-03               0.0            0.130135  menoufia  31.0  30.5      1   
2016-01-04               0

In [ ]:
# get_indicators(EGYPT_LOCATIONS, "egypt", ERA5)

UnboundLocalError: cannot access local variable 'FULL_DAYS' where it is not associated with a value

In [ ]:
import pandas as pd

monufia_df = pd.read_csv("../../cache/egypt_menoufia.csv")
behera_df = pd.read_csv("../../cache/egypt_beheira_damanhur.csv")
minya_df = pd.read_csv("../../cache/egypt_minya.csv")
sharqia_df = pd.read_csv("../../cache/egypt_sharqia.csv")
fayoum_df = pd.read_csv("../../cache/egypt_fayoum.csv")

In [ ]:
from config import get_settings
from pathlib import Path

app_settings = get_settings()
OUT_PATH = Path("data/egypt/egypt_soil.csv")
get_indicators(OUT_PATH, app_settings.EGYPT_LOCATIONS, "egypt")

ModuleNotFoundError: No module named 'ai_model'

In [11]:
import pandas as pd

df_old = pd.read_csv("../../data/egypt/egypt_climate_soil_datax.csv")

In [15]:
df_old.isna().sum()

date                     0
Temperature_C         7306
Temp_Min_C            7306
Temp_Max_C            7306
LST_Day_C               35
NDVI                     0
Precipitation_mm         0
Soil_Moisture_m3m3       0
location                 0
lon                      0
lat                      0
month                    0
day_of_year              0
dtype: int64

In [20]:
df_new = pd.read_csv("/data/egypt/egypt_climate_soil_data.csv")

In [21]:
df_new.isna().sum()

date                   0
Temperature_C          0
Temp_Min_C             0
Temp_Max_C             0
LST_Day_C             21
NDVI                   0
Precipitation_mm       0
Soil_Moisture_m3m3     0
location               0
lon                    0
lat                    0
month                  0
day_of_year            0
dtype: int64

In [2]:
import pandas as pd
df = pd.read_csv("/data/egypt/egypt_training_data.csv")

In [3]:
print(
    df.groupby("COMNAME")["irrigation_deficit_mm"]
    .mean()
    .sort_values(ascending=False)
    .round(0)
    .head(10)
)
print(
    df[["COMNAME", "ROPMN", "GMIN"]]
    .drop_duplicates()
    .sort_values("ROPMN", ascending=False)
    .head(10)
)

COMNAME
banana       2396.0
sunflower     254.0
garlic        246.0
wheat         246.0
sugarcane     211.0
orange        196.0
citrus        196.0
alfalfa       176.0
barley        163.0
potato        163.0
Name: irrigation_deficit_mm, dtype: float64
       COMNAME   ROPMN   GMIN
209     banana  2400.0    0.0
19   sugarcane  1500.0  210.0
152     orange  1200.0  180.0
133     citrus  1200.0  180.0
0        wheat   750.0   90.0
57      garlic   750.0   90.0
190      grape   700.0  160.0
171      mango   600.0  150.0
95     alfalfa   600.0  100.0
228  sunflower   600.0   70.0


In [11]:
df.isna().sum()

season                   0
season_year              0
COMNAME                  0
suitability_score        0
irrigation_deficit_mm    0
location                 0
lat                      0
lon                      0
t_mean                   0
t_min_mean               0
t_max_mean               0
t_min_abs                0
t_max_abs                0
ndvi_mean                0
sm_mean                  0
precip_total             0
TOPMN                    0
TOPMX                    0
TMIN                     0
TMAX                     0
KTMP                     0
GMIN                     0
ROPMN                    0
FAMNAME                  0
season_is_winter         0
date                     0
dtype: int64

In [8]:
ind = pd.read_csv("/data/egypt/egypt_climate_soil_data.csv", parse_dates=["date"])
cols = ["Temperature_C", "NDVI", "Precipitation_mm", "Soil_Moisture_m3m3"]
print(
    ind.groupby("location")[cols]
    .apply(lambda g: g.notna().mean())
    .round(2)
    .loc[["menoufia", "sharqia", "gharbia_tanta", "beheira_damanhur"]]
)

KeyError: "['gharbia_tanta'] not in index"

In [9]:
top = (
    df.groupby(["location", "season", "COMNAME"])["suitability_score"]
    .mean()
    .reset_index()
    .sort_values("suitability_score", ascending=False)
    .groupby(["location", "season"])
    .head(5)
)
for loc in ["beheira_damanhur", "luxor", "new_valley_kharga"]:
    print(loc)
    print(
        top[top.location == loc]
        .sort_values(["season", "suitability_score"], ascending=[True, False])[
            ["season", "COMNAME", "suitability_score"]
        ]
        .round(1)
        .to_string(index=False),
        "\n",
    )

beheira_damanhur
season   COMNAME  suitability_score
summer sugarcane               64.0
summer date palm               64.0
summer sunflower               60.4
summer     mango               52.3
summer    citrus               46.5
winter sunflower               60.9
winter    barley               57.5
winter    garlic               57.2
winter    potato               56.8
winter     grape               54.4 

luxor
season   COMNAME  suitability_score
summer date palm               64.4
summer sugarcane               61.8
summer sunflower               55.5
summer     mango               47.4
summer    citrus               38.9
winter sunflower               63.8
winter     grape               60.8
winter    garlic               59.7
winter   alfalfa               57.5
winter    orange               56.7 

new_valley_kharga
season   COMNAME  suitability_score
summer date palm               72.6
summer     mango               33.1
summer sunflower               31.6
summer sugarcane   

In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error as mae, r2_score
from sklearn.model_selection import GroupKFold

In [13]:


def new_model():
    return HistGradientBoostingRegressor(
        loss="absolute_error",
        max_iter=600,
        learning_rate=0.05,
        max_leaf_nodes=63,
        l2_regularization=1.0,
        random_state=42,
    )


def train():
    df = make_features(pd.read_csv("data/rotation_training_data.csv"))
    crops = load_crops()

    years = sorted(df["season_year"].unique())
    test = df["season_year"] == years[-1]
    m = new_model().fit(df.loc[~test, FEATURES], df.loc[~test, "suitability_score"])
    pred = m.predict(df.loc[test, FEATURES])
    print(
        f"Validation on {years[-1]}: MAE={mean_absolute_error(df.loc[test, 'suitability_score'], pred):.2f}  "
        f"R2={r2_score(df.loc[test, 'suitability_score'], pred):.3f}"
    )

    model = new_model().fit(df[FEATURES], df["suitability_score"])  # final: all data

    clim = df.drop_duplicates(["location", "season", "season_year"])[
        ["location", "season", "season_year", "lat", "lon"] + CLIMATE_FEATURES
    ]
    climatology = clim.groupby(["location", "season"], as_index=False)[
        ["lat", "lon"] + CLIMATE_FEATURES
    ].mean()

    Path(MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"model": model, "crops": crops, "climatology": climatology}, MODEL_PATH
    )
    print(f"Saved {MODEL_PATH}")


def score_crops(bundle, feats, season):
    crops = bundle["crops"].copy()
    for k in ["lat", "lon"] + CLIMATE_FEATURES:
        crops[k] = feats[k]
    crops["season_is_winter"] = int(season == "winter")
    crops = make_features(crops)
    crops["suitability"] = np.clip(bundle["model"].predict(crops[FEATURES]), 0, 100)
    crops["deficit"] = crops["formula_deficit"]  # exact formula, no ML needed
    crops["fits_season"] = crops["GMIN"].isna() | (crops["GMIN"] <= SEASON_DAYS[season])
    return crops